In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.sets.reachable_set import approximate_reachable_set
from dmpe.utils.sets.control_invariant_set import approximate_control_invariant_set

from dmpe.utils.sets.reachable_set import save_results as save_results_rs
from dmpe.utils.sets.control_invariant_set import save_results as save_results_ci
from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet, load_discretized_set

# Reachable set:

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()
key = jax.random.PRNGKey(14)
key, key_rs, _ = jax.random.split(key, 3)

obs_dim = env.reset(env.env_properties)[0].shape[-1]
sequence_length = 200
n_starts = 10
n_opt_steps = 50_000

tolerance = 1e-4

points_per_dim = 10
# target_observations_rs = build_grid(4, -1, 1, 15)

xs = [jnp.linspace(-1.0, 1.0, 25), jnp.linspace(-1.0, 1.0, 25), jnp.linspace(-1.0, 1.0, 25), jnp.linspace(-1.0, 1.0, 25)]
z_g = jnp.meshgrid(*xs, indexing="ij")
z_g = jnp.stack([_x for _x in z_g], axis=-1)
unflattened_shape = z_g.shape[:-1]

target_observations_rs = z_g.reshape(-1, obs_dim)

chunk_size = 20_000
n_targets = target_observations_rs.shape[0]

sets = []

print(n_targets / chunk_size  * 90 / 60)

for i in jnp.arange(0, n_targets, chunk_size):

    target_obs = target_observations_rs[i:min(i+chunk_size, n_targets)]
    print(target_obs.shape)

    R_s, debug_extras = approximate_reachable_set(
        env,
        target_obs,
        penalty_function,
        featurize,
        key=key_rs,
        sequence_length=sequence_length,
        n_starts=n_starts,
        n_opt_steps=n_opt_steps,
        tolerance=tolerance,
        unflattened_shape=unflattened_shape,
    )

    sets.append(R_s)

In [ ]:
grid = jnp.concatenate([rs.grid for rs in sets])
mask = jnp.concatenate([rs.mask for rs in sets])

full_set = DiscretizedSet(
    grid, mask, sets[0].unflattened_shape,
)

In [ ]:
from dmpe.utils.sets.reachable_set import save_reachable_set, load_reachable_set

In [ ]:
# save_reachable_set(
#     DataPaths().reach_ci_experiments / "cart_pole_sliced_reachable_set_1.json",
#     R_s=full_set,
# )

In [ ]:
loaded_R_s = load_reachable_set(DataPaths().reach_ci_experiments / "cart_pole_sliced_reachable_set.json",)
loaded_R_s.visualize(use_contourf=False)

---

In [ ]:
full_set.visualize()

In [ ]:
target_observations_rs.shape[0]

In [ ]:
# save_results_rs(
#     DataPaths().reach_ci_experiments / "cart_pole_reachabiltity_full_length_longer_training.json",
#     debug_extras[0],
#     debug_extras[1],
#     target_observations_rs,
# )

In [ ]:
R_s.visualize()

In [ ]:
from dmpe.utils.sets.reachable_set import loss_function as rs_loss_function 

In [ ]:
chosen_actions = debug_extras[0]

In [ ]:
chosen_actions.shape[0]

In [ ]:
init_obs, _ = env.reset(env.env_properties)
start=22222
for i in jnp.arange(start, start+20, 1):
    observations, _ = simulate_ahead_with_env(
        env,
        init_obs,
        env.generate_state_from_observation(init_obs, env.env_properties),
        chosen_actions[i]
    )
    loss_value = rs_loss_function(
        chosen_actions[i],
        target_observations_rs[i],
        init_obs,
        penalty_function,
        featurize,
        env,
    )
    print("loss: ", loss_value)
    if loss_value > 1e-3:
        #plot_sequence(observations, chosen_actions[i], env.tau, env.obs_description, env.action_description)
        # plt.show()
        labels = env.obs_description
        chosen_action_sequence = chosen_actions[i]
        observation_sequence = observations
        target = target_observations_rs[i]
    
        dim = observations.shape[-1]
    
        fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(9, 9), sharex=True, sharey=True)
        for i in range(dim):
            for j in range(dim):
                axs[j, i].grid(True)
                axs[j, i].plot(target[i], target[j], 'rx')
                axs[j, i].plot(observation_sequence[..., i], observation_sequence[..., j])
                axs[j, 0].set_ylabel(labels[j])
            axs[-1, i].set_xlabel(labels[i])
        fig.tight_layout()
        plt.show()
        print("")

In [ ]:
R_s

In [ ]:
# save_results_rs(
#     DataPaths().reach_ci_experiments / "cart_pole_reachabiltity.json",
#     chosen_actions_rs,
#     losses_rs,
#     target_observations_rs,
# )

In [ ]:
# env, penalty_function, featurize, _ = setup_cart_pole_env()
# data_rs = load_results(
#     DataPaths().reach_ci_experiments / "cart_pole_reachabiltity.json",
# )

In [ ]:
data_rs = dict(
    chosen_actions_rs=chosen_actions_rs,
    losses_rs=losses_rs,
    target_observations_rs=target_observations_rs,
)

In [ ]:
points_per_dim = 15
dim = 4

In [ ]:
safe = (data_rs["losses_rs"] < 1e-3).reshape([points_per_dim] * dim)

In [ ]:
labels = env.obs_description
n_features = 4
fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, n_features, 1).tolist()

for i in range(n_features):
    for j in range(n_features):

        axs[j, i].grid(True)
        axs[j, i].set_xlim(-1.1, 1.1)
        axs[j, i].set_ylim(-1.1, 1.1)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == n_features - 1:
            continue

        any_safe = jnp.any(safe, axis=tuple(reduction_indices))

        if i < j:
            any_safe = jnp.transpose(any_safe)

        axs[j, i].imshow(any_safe, origin="lower", extent=[-1, 1, -1, 1])
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()

- transfer to actual set representation? see [Baier2012]
- I guess this would be a function that evaluates an input to a bool?

# Control invariant set:

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()
key = jax.random.PRNGKey(14)
key, _, key_ci = jax.random.split(key, 3)

obs_dim = env.reset(env.env_properties)[0].shape[-1]
sequence_length = 500
n_starts = 100
n_opt_steps = 20_000

xs = [jnp.linspace(-1.0, 1.0, 50), jnp.linspace(0.8, 1.0, 50)]
z_g = jnp.meshgrid(*xs, indexing="ij")
z_g = jnp.stack([_x for _x in z_g], axis=-1)
init_observations_ci = z_g.reshape(-1, 2)

chosen_actions_ci, losses_ci, proposed_actions_ci = approximate_control_invariant_set(
    env,
    init_observations_ci,
    penalty_function,
    key=key_ci,
    sequence_length=sequence_length,
    n_starts=n_starts,
    n_opt_steps=n_opt_steps,
)

points_per_dim = int(jnp.sqrt(init_observations_ci.shape[0]))
plt.contourf(
     init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
     init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
     jnp.abs(losses_ci.reshape(points_per_dim, points_per_dim)),
)
plt.show()

In [ ]:
save_results_ci(
    DataPaths().reach_ci_experiments / "cart_pole_control_invariance.json",
    chosen_actions_ci,
    losses_ci,
    init_observations_ci,
)

In [ ]:
data_rs = load_results_ci(
    DataPaths().reach_ci_experiments / "cart_pole_control_invariance.json",
)

# Combine:

In [ ]:
data_ci = load_results(
    DataPaths().reach_ci_experiments / "cart_pole_control_invariance_full_length_longer_training.json",
)

data_rs = load_results(
    DataPaths().reach_ci_experiments / "cart_pole_reachabiltity_full_length_longer_training.json",
)

In [ ]:
dim = 4
points_per_dim = 15

test_C = DiscretizedSet(
    grid=data_ci["init_observations_ci"],
    mask=jnp.isclose(jnp.abs(data_ci["losses_ci"]), 0),
    unflattened_shape=tuple([points_per_dim] * dim),
)

test_Rs = DiscretizedSet(
    grid=data_rs["target_observations_rs"],
    mask=jnp.abs(data_rs["losses_rs"]) < 1e-5,
    unflattened_shape=tuple([points_per_dim] * dim),
)

In [ ]:
test_C.visualize()
plt.show()

test_Rs.visualize()
plt.show()

In [ ]:
S = test_C & test_Rs

In [ ]:
S.visualize()

# C(x) to C(x, u)